# Deep learning: fine-tuned DistilBERT (notebook 03)

The whole pretrained encoder is fine-tuned on the training rows and scored once on test per seed,
on the same five splits as notebook 02; comparing and plotting the saved results is notebook 04.
The run cell needs a GPU and the `torch`, `transformers` and `captum` packages, and is skipped
unless `ON_GPU_SERVER = True` (set it on the server only).

Run time: about 5 minutes per seed on one GPU at batch size 128 (5 epochs).

In [ ]:
import sys, os; os.chdir(os.path.join(os.getcwd(), "..")) if os.path.basename(os.getcwd()) == "notebooks" else None; sys.path.insert(0, "."); import warnings
import importlib.util

from modules import data_helper as dh
from modules import features as ft
from modules import deep_runs as dr

In [ ]:
SEED = 0
SEEDS = [0, 1, 2, 3, 4]  # seeds with existing result files are skipped
OUT_DEEP = "outputs/deep"

BERT_MODEL = "distilbert-base-uncased"
EPOCHS, BATCH_SIZE, MAX_LEN, LR = 5, 128, 256, 2e-5
LOG_EVERY = 10  # print the loss every this many steps
DEVICE = "cuda:1"  # which GPU to use; the server is shared, so pick an emptier one (nvidia-smi)

# SAFETY: fine-tuning is heavy GPU work. It runs only when this flag is True, and the flag must
# stay False on the laptop. Set it to True on the GPU server, nowhere else.
ON_GPU_SERVER = False

# Libraries present? (a missing one would make the run cell fail, not skip)
HAVE_TORCH = all(importlib.util.find_spec(m) for m in ("torch", "transformers", "captum"))
RUN_BERT = ON_GPU_SERVER and HAVE_TORCH
print({"ON_GPU_SERVER": ON_GPU_SERVER, "torch stack": HAVE_TORCH})

dh.seed_everything(SEED)

## Data and splits

The same processed data and the same saved splits as the classical notebook, so every model is
scored on identical rows.

In [ ]:
df = ft.add_engineered_features(dh.load_processed())
print("rows:", len(df), "| fraud:", int(df["fraudulent"].sum()))

## Fine-tune DistilBERT

The whole encoder is trained with a linear classification layer on top: AdamW, learning rate
2e-5 with warm-up and linear decay, square-root class weights in the loss, bf16 autocast on
CUDA, and early stopping on validation PR-AUC (the best epoch's weights are restored). Texts
are truncated to 256 tokens.

Model explanation: right after seed 0 is trained, Integrated Gradients scores how much each token
pushed the model towards fraud, for 10 seed-0 test ads (4 caught frauds, 3 missed frauds, 3 false
alarms). They are saved next to the results and shown in notebook 04.

In [ ]:
if RUN_BERT:
    dr.run_bert(df, SEEDS, OUT_DEEP, DEVICE, BERT_MODEL, epochs=EPOCHS,
                batch_size=BATCH_SIZE, max_len=MAX_LEN, lr=LR, log_every=LOG_EVERY)
else:
    print("skipped: set ON_GPU_SERVER = True (GPU server only) and install torch + transformers + captum")